# 03 · Load flights
Loads every `flights_*.csv` into `local.airline.flights`, **partitioned by `Year`**.

* Dropped and recreated each run (idempotent).
* `inferSchema` makes two passes over the CSVs. That is fine for a laptop-sized extract but slow for
  very large files; for those, define an explicit schema instead.

In [ ]:
from pyspark.sql import SparkSession

# Catalog config (REST catalog + MinIO) comes from spark-defaults.conf.
spark = SparkSession.builder.appName("03-flights").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

def drop_table(name):
    """Drop an Iceberg table and its data files; fall back to a plain drop."""
    try:
        spark.sql(f"DROP TABLE IF EXISTS {name} PURGE")
    except Exception:
        spark.sql(f"DROP TABLE IF EXISTS {name}")

from pathlib import Path
from pyspark.sql import functions as F

DATA_DIR = Path.home() / "data"
TABLE = "local.airline.flights"
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.airline")

In [ ]:
files = sorted(DATA_DIR.glob("flights_*.csv"))
if not files:
    raise FileNotFoundError(f"no flights_*.csv in {DATA_DIR}. See README, 'Getting the data'.")
print(f"{len(files)} file(s):", ", ".join(f.name for f in files))

df = spark.read.csv([str(f) for f in files], header=True, inferSchema=True, nullValue="NA")
assert "Year" in df.columns, f"expected a 'Year' column to partition by; got {df.columns}"
df.printSchema()

In [ ]:
drop_table(TABLE)
df.writeTo(TABLE).partitionedBy(F.col("Year")).create()
print(f"✅ {TABLE}: {spark.table(TABLE).count():,} rows")

In [ ]:
spark.sql(f"SELECT partition, record_count, file_count FROM {TABLE}.partitions ORDER BY partition").show()